Question 1 (a)

In [1]:
import numpy as np

In [2]:
# Load data
data = np.loadtxt('lines.csv', delimiter=',', comments='#')

x1 = data[:, 0]
y1 = data[:, 3]

# --- Total Least Squares (TLS) via SVD ---

# Step 1: Center the data
x1_mean = np.mean(x1)
y1_mean = np.mean(y1)

# Step 2: Form the centered data matrix
X = np.column_stack([x1 - x1_mean, y1 - y1_mean])

# Step 3: SVD of the centered matrix
U, S, Vt = np.linalg.svd(X, full_matrices=False)

# Step 4: The last row of Vt (smallest singular value) is the normal vector [a, b]
v = Vt[-1]  # normal vector to the line

# Step 5: Convert to slope-intercept form  y = slope * x + intercept
slope     = -v[0] / v[1]
intercept = y1_mean - slope * x1_mean

print(f"Slope:     {slope:.6f}")
print(f"Intercept: {intercept:.6f}")

Slope:     1.220666
Intercept: -5.987165


Question 1 (b)

In [3]:
# ── Data loading exactly as specified ─────────────────────────────────────────
D      = np.genfromtxt("lines.csv", delimiter=",", skip_header=1)
X_cols = D[:, :3]
Y_cols = D[:, 3:]
X_all  = X_cols.flatten()
Y_all  = Y_cols.flatten()

# ── TLS helper (SVD-based) ────────────────────────────────────────────────────
def fit_tls(x, y):
    """Fit a line y = slope*x + intercept using Total Least Squares."""
    x_mean, y_mean = np.mean(x), np.mean(y)
    M = np.column_stack([x - x_mean, y - y_mean])
    _, _, Vt = np.linalg.svd(M, full_matrices=False)
    a, b = Vt[-1]          # normal vector (smallest singular vector)
    slope     = -a / b
    intercept = y_mean - slope * x_mean
    return slope, intercept

def perpendicular_distances(x, y, slope, intercept):
    """Perpendicular distance from each point to line: slope*x - y + intercept = 0"""
    return np.abs(slope * x - y + intercept) / np.sqrt(slope**2 + 1)

# ── RANSAC with TLS ───────────────────────────────────────────────────────────
def ransac_tls(x, y, n_iter=2000, sample_size=2, inlier_threshold=0.5, seed=42):
    rng = np.random.default_rng(seed)
    best_inliers = None
    best_count   = 0

    for _ in range(n_iter):
        idx = rng.choice(len(x), size=sample_size, replace=False)
        s, i = fit_tls(x[idx], y[idx])
        dists = perpendicular_distances(x, y, s, i)
        inlier_mask = dists < inlier_threshold
        if inlier_mask.sum() > best_count:
            best_count   = inlier_mask.sum()
            best_inliers = inlier_mask

    # Refit using ALL inliers with TLS
    slope, intercept = fit_tls(x[best_inliers], y[best_inliers])
    return slope, intercept, best_inliers

# ── Find 3 lines sequentially (mask consensus, repeat) ───────────────────────
remaining_mask = np.ones(len(X_all), dtype=bool)

for line_num in range(1, 4):
    x = X_all[remaining_mask]
    y = Y_all[remaining_mask]

    slope, intercept, inlier_mask_local = ransac_tls(x, y)
    slope, intercept = fit_tls(x[inlier_mask_local], y[inlier_mask_local])

    print(f"Line {line_num}: slope = {slope:.6f}, intercept = {intercept:.6f}, "
          f"inliers = {inlier_mask_local.sum()}")

    # Remove consensus inliers from the pool
    global_indices = np.where(remaining_mask)[0]
    remaining_mask[global_indices[inlier_mask_local]] = False

Line 1: slope = -0.454436, intercept = 2.087364, inliers = 84
Line 2: slope = 1.034381, intercept = 1.029066, inliers = 67
Line 3: slope = 1.242646, intercept = -6.174927, inliers = 65


Question 2

In [5]:
from PIL import Image

# ── Step 1: Load image and create non-white mask ───────────────────────────────
img  = Image.open("earrings.jpg")          # <-- change to your filename
arr  = np.array(img)
mask = np.any(arr < 220, axis=2)           # True where pixel is non-white

# ── Step 2: Find the two earrings by splitting on the vertical gap ─────────────
# Column profile: count non-white pixels per column
col_profile = mask.sum(axis=0)

# Find columns that belong to earring regions (non-zero) vs gap (zero)
non_empty_cols = np.where(col_profile > 0)[0]
c_start = non_empty_cols[0]
c_end   = non_empty_cols[-1]

# Find the gap between the two earrings (columns with 0 non-white pixels)
# within the earring column range
mid_region = col_profile[c_start:c_end+1]
gap_indices = np.where(mid_region == 0)[0] + c_start

# Split point = middle of the gap
gap_mid = int((gap_indices[0] + gap_indices[-1]) / 2)

# ── Step 3: Measure each earring separately ────────────────────────────────────
def measure_earring(mask_region):
    """Return (width_px, height_px) of the largest object in a mask region."""
    rows = np.where(mask_region.any(axis=1))[0]
    cols = np.where(mask_region.any(axis=0))[0]
    
    # Filter to only the earring rows (ignore tiny text blobs at bottom)
    # Use row profile: keep only rows with significant content
    row_profile = mask_region.sum(axis=1)
    row_threshold = row_profile.max() * 0.05   # at least 5% of max density
    earring_rows = np.where(row_profile > row_threshold)[0]
    
    # Recompute with filtered rows
    earring_mask = mask_region[earring_rows, :]
    filtered_cols = np.where(earring_mask.any(axis=0))[0]
    
    h_px = earring_rows[-1] - earring_rows[0] + 1
    w_px = filtered_cols[-1] - filtered_cols[0] + 1
    return w_px, h_px

left_mask  = mask[:, c_start:gap_mid]
right_mask = mask[:, gap_mid:c_end+1]

left_w,  left_h  = measure_earring(left_mask)
right_w, right_h = measure_earring(right_mask)

print(f"Earring 1: width={left_w} px,  height={left_h} px")
print(f"Earring 2: width={right_w} px, height={right_h} px")

avg_w_px = (left_w  + right_w)  / 2
avg_h_px = (left_h  + right_h)  / 2
diam_px  = (avg_w_px + avg_h_px) / 2

print(f"\nAverage: width={avg_w_px:.1f} px, height={avg_h_px:.1f} px")
print(f"Diameter estimate: {diam_px:.1f} px")

# ── Step 4: Camera / thin-lens model ──────────────────────────────────────────
f = 8.0      # mm  focal length
u = 720.0    # mm  object distance (lens to earring plane)
s = 2.2e-3   # mm  pixel size (2.2 µm)

# Thin-lens equation: 1/f = 1/v + 1/u  →  v = f*u / (u-f)
v = (f * u) / (u - f)    # image distance in mm
m = v / u                 # lateral magnification

print(f"\nImage distance  v = {v:.4f} mm")
print(f"Magnification   m = {m:.6f}")

# ── Step 5: Pixel → real-world size ───────────────────────────────────────────
obj_w_mm = (avg_w_px * s) / m
obj_h_mm = (avg_h_px * s) / m
obj_d_mm = (diam_px  * s) / m

print(f"\n{'='*45}")
print(f"Real-world earring size:")
print(f"  Width    = {obj_w_mm:.2f} mm")
print(f"  Height   = {obj_h_mm:.2f} mm")
print(f"  Diameter = {obj_d_mm:.2f} mm")
print(f"{'='*45}")

Earring 1: width=382 px,  height=396 px
Earring 2: width=382 px, height=396 px

Average: width=382.0 px, height=396.0 px
Diameter estimate: 389.0 px

Image distance  v = 8.0899 mm
Magnification   m = 0.011236

Real-world earring size:
  Width    = 74.80 mm
  Height   = 77.54 mm
  Diameter = 76.17 mm
